# Проверка API сомелье

Запустите сервис через `docker compose up --build -d` из корня проекта. Затем выполняйте ячейки по порядку. Ключ OpenRouter в notebook не нужен.

In [1]:
import secrets
import time
import uuid

import httpx

In [2]:
BASE_URL = "http://127.0.0.1:8000"

In [3]:
client = httpx.Client(base_url=BASE_URL, timeout=90.0, trust_env=False)


def show(response: httpx.Response):
    print(f"{response.request.method} {response.request.url} → {response.status_code}")
    if response.content:
        payload = response.json()
        display(payload)
    else:
        payload = None
    response.raise_for_status()
    return payload

In [4]:
show(client.get("/health"))

GET http://127.0.0.1:8000/health → 200


{'status': 'ok'}

{'status': 'ok'}

## Новый диалог

Примеры `wine` и `candidates` взяты из каталога соседнего репозитория `vinishko/data/catalog/catalog.csv`: Каберне и похожие карточки той же серии. При необходимости замените их своими. Каждое выполнение следующей ячейки создаёт новый UUIDv7 и новую сессию.

In [5]:
def new_uuid7() -> uuid.UUID:
    # Python 3.13 ещё не предоставляет uuid.uuid7().
    timestamp_ms = time.time_ns() // 1_000_000
    random_bits = secrets.randbits(74)
    value = (timestamp_ms & ((1 << 48) - 1)) << 80
    value |= 7 << 76
    value |= ((random_bits >> 62) & 0xFFF) << 64
    value |= 0b10 << 62
    value |= random_bits & ((1 << 62) - 1)
    return uuid.UUID(int=value)

In [6]:
session_id = str(new_uuid7())
wine = {
    "Название вина": "100 оттенков красного. Каберне",
    "Категория": "Красное",
    "Цвет": "Темно-рубиновый",
    "Регион": "Кубань",
    "Сорт винограда": "Каберне Совиньон",
    "Описание": "В аромате спелая вишня, чёрная смородина, лёгкая пряность. Вкус плотный, гармоничный, с мягкими танинами и ягодным послевкусием.",
    "Винодельня": "Фанагория",
    "Slug": "fanagoriya-100-ottenkov-krasnogo-kaberne-kaberne-sovinon-krasnoe-suhoe-135",
    "Крепость, %": 13.5,
    "Сахар": "сухое",
}
candidates = [
    {
        "Название вина": "100 оттенков красного. Пино Нуар",
        "Категория": "Красное",
        "Регион": "Кубань",
        "Сорт винограда": "Пино Нуар",
        "Описание": "Стройный развитый аромат с нотами брусники, вяленых лесных ягод, гранатовых зерен, пастилы, тонкими животными нюансами. Вкус легкий и волнующий, многослойный. Все элементы вина находятся в балансе. Тонко, свежо, элегантно.",
        "Винодельня": "Фанагория",
        "Slug": "fanagoriya-100-ottenkov-krasnogo-pino-nuar-krasnoe-suhoe-135",
        "Крепость, %": 13.5,
        "Сахар": "сухое",
    },
    {
        "Название вина": "100 оттенков красного. Саперави",
        "Категория": "Красное",
        "Регион": "Кубань",
        "Сорт винограда": "Саперави",
        "Описание": "В аромате тона ежевики, черники, тёмной вишни, оттенки специй. Вкус насыщенный, сбалансированный, с бархатными танинами.",
        "Винодельня": "Фанагория",
        "Slug": "fanagoriya-100-ottenkov-krasnogo-saperavi-krasnoe-suhoe-14",
        "Крепость, %": 14,
        "Сахар": "сухое",
    },
]

In [7]:
opening = show(
    client.post(
        f"/v1/sessions/{session_id}",
        json={"wine": wine, "candidates": candidates},
    )
)
print("session_id:", session_id)
print("Подсказки:", opening["message"]["suggestions"])

POST http://127.0.0.1:8000/v1/sessions/01a0dd1d-00a3-7663-bcfa-20c00dec390a → 502


{'detail': 'Sommelier is temporarily unavailable'}

HTTPStatusError: Server error '502 Bad Gateway' for url 'http://127.0.0.1:8000/v1/sessions/01a0dd1d-00a3-7663-bcfa-20c00dec390a'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/502

In [ ]:
question = "С чем лучше сочетать это вино?"
reply = show(
    client.post(
        f"/v1/sessions/{session_id}/messages",
        json={"content": question},
    )
)

In [ ]:
history = show(client.get(f"/v1/sessions/{session_id}"))

## Необязательно: удалить сессию

Запустите следующую ячейку, когда история больше не нужна.

In [ ]:
show(client.delete(f"/v1/sessions/{session_id}"))